In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
movies = pd.read_csv(
    "../dataset/processed/movieverse_clean_dataset.csv"
)

movies.head()

C:\Users\HARIOM SUTHAR\AppData\Local\Temp\ipykernel_14792\2416910609.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv(


,movie_id,title,release_year,tagline,overview,runtime_minutes,avg_rating,genres,main_cast,poster_url,director,producer,languages,themes,countries,studios,decade
0,1000001,Barbie,2023.0,She's everything. He's just Ken.,Barbie and Ken are having the time of their li...,114.0,3.86,Comedy | Adventure,Margot Robbie as Barbie | Ryan Gosling as Ken ...,https://a.ltrbxd.com/resized/film-poster/2/7/7...,Greta Gerwig,Tom Ackerley | Margot Robbie | Robbie Brenner,English,Humanity and the world around us | Crude humor...,UK | USA,LuckyChap Entertainment | Heyday Films | NB/GG...,2020.0
1,1000002,Parasite,2019.0,Act like you own the place.,"All unemployed, Ki-taek's family takes peculia...",133.0,4.56,Comedy | Thriller | Drama,Song Kang-ho as Kim Ki-taek | Lee Sun-kyun as ...,https://a.ltrbxd.com/resized/film-poster/4/2/6...,Bong Joon-ho,Jang Young-hwan | Kwak Sin-ae | Moon Yang-kwon,Korean | English | German,Humanity and the world around us | Intense vio...,South Korea,Barunson E&A,2010.0
2,1000003,Everything Everywhere All at Once,2022.0,The universe is so much bigger than you realize.,An aging Chinese immigrant is swept up in an i...,140.0,4.30,Science Fiction | Adventure | Comedy | Action,Michelle Yeoh as Evelyn Wang | Ke Huy Quan as ...,https://a.ltrbxd.com/resized/film-poster/4/7/4...,Daniel Scheinert | Daniel Kwan,Joe Russo | Anthony Russo | Jonathan Wang,English | Cantonese | Chinese,Humanity and the world around us | Moving rela...,USA,IAC Films | AGBO | Ley Line Entertainment,2020.0
3,1000004,Fight Club,1999.0,Mischief. Mayhem. Soap.,A ticking-time-bomb insomniac and a slippery s...,139.0,4.27,Drama,Edward Norton as Narrator | Brad Pitt as Tyler...,https://a.ltrbxd.com/resized/film-poster/5/1/5...,David Fincher,Ross Grayson Bell | Ceán Chaffin | Art Linson,English,Intense violence and sexual transgression | Hu...,Germany | USA,Fox 2000 Pictures | Regency Enterprises | The ...,1990.0
4,1000005,La La Land,2016.0,Here's to the fools who dream.,"Mia, an aspiring actress, serves lattes to mov...",129.0,4.09,Drama | Comedy | Music | Romance,Ryan Gosling as Sebastian | Emma Stone as Mia ...,https://a.ltrbxd.com/resized/film-poster/2/4/0...,Damien Chazelle,Jordan Horowitz | Fred Berger | Marc Platt,English,Song and dance | Humanity and the world around...,Hong Kong | USA,Summit Entertainment | Black Label Media | Gil...,2010.0


In [3]:
print(movies.shape)

movies.info()

(570748, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570748 entries, 0 to 570747
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   movie_id         570748 non-null  int64  
 1   title            570740 non-null  object 
 2   release_year     514419 non-null  float64
 3   tagline          96105 non-null   object 
 4   overview         570748 non-null  object 
 5   runtime_minutes  389178 non-null  float64
 6   avg_rating       74215 non-null   float64
 7   genres           429424 non-null  object 
 8   main_cast        570748 non-null  object 
 9   poster_url       477317 non-null  object 
 10  director         463916 non-null  object 
 11  producer         570748 non-null  object 
 12  languages        570748 non-null  object 
 13  themes           23618 non-null   object 
 14  countries        570748 non-null  object 
 15  studios          570748 non-null  object 
 16  decade           514419 n

In [4]:
feature_columns = [
    "genres",
    "overview",
    "director",
    "main_cast",
    "themes",
    "languages",
    "countries",
    "studios"
]

movies[feature_columns] = (
    movies[feature_columns]
    .fillna("")
)

In [5]:
movies[feature_columns].isna().sum()

genres       0
overview     0
director     0
main_cast    0
themes       0
languages    0
countries    0
studios      0
dtype: int64

In [6]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [7]:
for col in feature_columns:
    movies[col] = movies[col].apply(clean_text)

In [8]:
movies[
    [
        "genres",
        "overview",
        "director",
        "main_cast"
    ]
].head()

,genres,overview,director,main_cast
0,comedy adventure,barbie and ken are having the time of their li...,greta gerwig,margot robbie as barbie ryan gosling as ken am...
1,comedy thriller drama,all unemployed ki taek s family takes peculiar...,bong joon ho,song kang ho as kim ki taek lee sun kyun as pa...
2,science fiction adventure comedy action,an aging chinese immigrant is swept up in an i...,daniel scheinert daniel kwan,michelle yeoh as evelyn wang ke huy quan as wa...
3,drama,a ticking time bomb insomniac and a slippery s...,david fincher,edward norton as narrator brad pitt as tyler d...
4,drama comedy music romance,mia an aspiring actress serves lattes to movie...,damien chazelle,ryan gosling as sebastian emma stone as mia jo...


### Weighted Features

In [9]:
def repeat_text(text, times):
    if text.strip() == "":
        return ""
    return (" " + text) * times

In [10]:
repeat_text("action adventure", 3)

' action adventure action adventure action adventure'

### Create Weighted Tags

In [11]:
movies["tags"] = (

    movies["genres"].apply(lambda x: repeat_text(x, 4))

    + movies["director"].apply(lambda x: repeat_text(x, 3))

    + movies["main_cast"].apply(lambda x: repeat_text(x, 3))

    + movies["themes"].apply(lambda x: repeat_text(x, 2))

    + movies["overview"].apply(lambda x: repeat_text(x, 2))

    + movies["languages"].apply(lambda x: repeat_text(x, 1))

    + movies["countries"].apply(lambda x: repeat_text(x, 1))

    + movies["studios"].apply(lambda x: repeat_text(x, 1))

)

In [14]:
movies[["title", "tags"]].head(10)

,title,tags
0,Barbie,comedy adventure comedy adventure comedy adven...
1,Parasite,comedy thriller drama comedy thriller drama co...
2,Everything Everywhere All at Once,science fiction adventure comedy action scienc...
3,Fight Club,drama drama drama drama david fincher david fi...
4,La La Land,drama comedy music romance drama comedy music ...
5,Oppenheimer,drama history drama history drama history dram...
6,Interstellar,science fiction drama adventure science fictio...
7,Joker,crime drama thriller crime drama thriller crim...
8,Dune,science fiction adventure science fiction adve...
9,Pulp Fiction,crime thriller crime thriller crime thriller c...


In [13]:
movies["tags"] = movies["tags"].str.replace(r"\s+", " ", regex=True).str.strip()

In [15]:
movies["tags"].eq("").sum()

np.int64(0)

In [16]:
movies["tags"] = movies["tags"].str.split()

movies["tags"].head()

0    [comedy, adventure, comedy, adventure, comedy,...
1    [comedy, thriller, drama, comedy, thriller, dr...
2    [science, fiction, adventure, comedy, action, ...
3    [drama, drama, drama, drama, david, fincher, d...
4    [drama, comedy, music, romance, drama, comedy,...
Name: tags, dtype: object

In [17]:
def remove_extra_duplicates(words, max_repeat=3):
    
    word_count = {}
    result = []

    for word in words:

        count = word_count.get(word, 0)

        if count < max_repeat:
            result.append(word)

        word_count[word] = count + 1

    return result

In [18]:
movies["tags"] = movies["tags"].apply(remove_extra_duplicates)

In [19]:
movies["tags"] = movies["tags"].apply(" ".join)

In [20]:
movies.loc[
    0,
    ["title", "genres", "director", "main_cast", "tags"]
]

title                                                   Barbie
genres                                        comedy adventure
director                                          greta gerwig
main_cast    margot robbie as barbie ryan gosling as ken am...
tags         comedy adventure comedy adventure comedy adven...
Name: 0, dtype: object

In [21]:
print("Empty Tags :", (movies["tags"] == "").sum())

Empty Tags : 0


In [22]:
movies.to_csv(
    "../dataset/processed/movieverse_feature_engineered.csv",
    index=False
)

print("Feature Engineered Dataset Saved")
print(movies.shape)

Feature Engineered Dataset Saved
(570748, 18)
